In [1]:
!pip install pyvi kagglehub transformers torch scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 58.9 MB/s eta 0:00:00


In [2]:
import kagglehub
import pandas as pd
import os
import numpy as np

print("=== QUY TRÌNH GỘP 4 DATASET (FINAL VERSION) ===")
print("Quy ước: 0 = REAL (Tin thật), 1 = FAKE (Tin giả)")

df_list = []

# ==============================================================================
# 1. DATASET 1: phngnguynthu1803
# ==============================================================================
print("\n1. Đang xử lý Dataset 1 (phngnguynthu1803)...")
try:
    path1 = kagglehub.dataset_download("phngnguynthu1803/vietnamese-fake-news-dataset")
    for file in os.listdir(path1):
        if file.endswith('.csv'):
            try:
                temp = pd.read_csv(os.path.join(path1, file))
                temp.rename(columns={'news': 'text', 'content': 'text'}, inplace=True)

                # Logic xử lý nhãn dựa trên tên file
                if 'real' in file.lower():
                    temp['label'] = 0
                elif 'fake' in file.lower():
                    temp['label'] = 1
                else:
                    # Fallback
                    temp['label'] = temp['label'].apply(lambda x: 0 if x == 1 else 1)

                if 'text' in temp.columns:
                    df_list.append(temp[['text', 'label']])
            except: pass
except Exception as e: print(f"❌ Lỗi Dataset 1: {e}")

# ==============================================================================
# 2. DATASET 2: chuynvinquc
# ==============================================================================
print("\n2. Đang xử lý Dataset 2 (chuynvinquc)...")
try:
    path2 = kagglehub.dataset_download("chuynvinquc/fakenewvn")
    csv_files = [f for f in os.listdir(path2) if f.endswith('.csv')]
    if csv_files:
        temp = pd.read_csv(os.path.join(path2, csv_files[0]))
        temp.rename(columns={'post_message': 'text', 'label': 'label'}, inplace=True)
        if 'text' in temp.columns:
            df_list.append(temp[['text', 'label']])
            print(f"   - Đã nạp: {csv_files[0]}")
except Exception as e: print(f"❌ Lỗi Dataset 2: {e}")

# ==============================================================================
# 3. DATASET 3: goumanguyen
# ==============================================================================
print("\n3. Đang xử lý Dataset 3 (goumanguyen)...")
try:
    path3 = kagglehub.dataset_download("goumanguyen/vietnamese-fake-news-dataset-pbl7")
    for root, dirs, files in os.walk(path3):
        for file in files:
            if file.endswith(".csv"):
                try:
                    temp = pd.read_csv(os.path.join(root, file))
                    temp.rename(columns={'Maintext': 'text', 'Label': 'label', 'maintext': 'text'}, inplace=True)
                    if 'text' in temp.columns and 'label' in temp.columns:
                        if temp['label'].dtype == 'object':
                            temp['label'] = temp['label'].map({'Real': 0, 'Fake': 1, 'TIN THẬT': 0, 'TIN GIẢ': 1})
                        temp['label'] = pd.to_numeric(temp['label'], errors='coerce')
                        temp.dropna(subset=['label'], inplace=True)
                        temp['label'] = temp['label'].astype(int)
                        df_list.append(temp[['text', 'label']])
                        print(f"   - Đã nạp: {file}")
                except: pass
except Exception as e: print(f"❌ Lỗi Dataset 3: {e}")

# ==============================================================================
# [cite_start]4. DATASET 4: leviettrieu369 (Dataset Y tế mới) [cite: 5, 10]
# Yêu cầu: is_fake -> label (True=1, False=0), normalized_content -> text
# ==============================================================================
print("\n4. Đang xử lý Dataset 4 (leviettrieu369 - Medical)...")
try:
    path4 = kagglehub.dataset_download("leviettrieu369/vietnamese-medical-fake-news-dataset")
    csv_files = [f for f in os.listdir(path4) if f.endswith('.csv')]

    if csv_files:
        file_path = os.path.join(path4, csv_files[0])
        temp = pd.read_csv(file_path)

        # 1. Đổi tên cột (Lấy normalized_content làm text chính)
        temp.rename(columns={'normalized_content': 'text', 'is_fake': 'label'}, inplace=True)

        # 2. Map nhãn (True -> 1, False -> 0)
        # Xử lý cả kiểu boolean và kiểu chuỗi đề phòng
        mapping = {
            True: 1, False: 0,
            'True': 1, 'False': 0,
            'TRUE': 1, 'FALSE': 0
        }

        if 'label' in temp.columns:
            temp['label'] = temp['label'].map(mapping)

            # Xóa các dòng NaN (nếu có nhãn lạ)
            temp.dropna(subset=['label', 'text'], inplace=True)
            temp['label'] = temp['label'].astype(int)

            # Chỉ lấy 2 cột chuẩn
            if 'text' in temp.columns:
                df_list.append(temp[['text', 'label']])
                print(f"   - Đã nạp: {csv_files[0]} ({len(temp)} dòng)")
                print(f"   - Phân bố nhãn Dataset 4: {temp['label'].value_counts().to_dict()}")
        else:
            print(f"   ⚠️ Không tìm thấy cột 'is_fake' (đã rename thành label). Cột hiện có: {list(temp.columns)}")

except Exception as e: print(f"❌ Lỗi Dataset 4: {e}")


# ==============================================================================
# TỔNG HỢP CUỐI CÙNG
# ==============================================================================
print("\n...Đang trộn và làm sạch toàn bộ dữ liệu...")
if df_list:
    df_final = pd.concat(df_list, ignore_index=True)

    # Xóa dòng trùng lặp nội dung
    df_final.drop_duplicates(subset=['text'], keep='first', inplace=True)

    # Xóa dòng quá ngắn (Rác)
    df_final = df_final[df_final['text'].str.len() > 20]

    # Xáo trộn ngẫu nhiên
    df = df_final.sample(frac=1, random_state=42).reset_index(drop=True)

    print("-" * 40)
    print("🎉 KẾT QUẢ CUỐI CÙNG (DATASET KHỔNG LỒ):")
    counts = df['label'].value_counts()
    print(counts)
    print(f"-> Tỉ lệ: {counts.get(0, 0)} Tin Thật vs {counts.get(1, 0)} Tin Giả")
    print("-" * 40)
    print(f"Tổng cộng: {len(df)} dòng dữ liệu.")
else:
    print("❌ Không gộp được dữ liệu nào!")

=== QUY TRÌNH GỘP 4 DATASET (FINAL VERSION) ===
Quy ước: 0 = REAL (Tin thật), 1 = FAKE (Tin giả)

1. Đang xử lý Dataset 1 (phngnguynthu1803)...

2. Đang xử lý Dataset 2 (chuynvinquc)...
   - Đã nạp: public_train.csv

3. Đang xử lý Dataset 3 (goumanguyen)...
   - Đã nạp: fix_test_data.csv
   - Đã nạp: update_val_data.csv
   - Đã nạp: update_train_data.csv
   - Đã nạp: val_data.csv
   - Đã nạp: train_data.csv
   - Đã nạp: fix_test_data.csv

4. Đang xử lý Dataset 4 (leviettrieu369 - Medical)...
   - Đã nạp: full_dataset.csv (10617 dòng)
   - Phân bố nhãn Dataset 4: {0: 5692, 1: 4925}

...Đang trộn và làm sạch toàn bộ dữ liệu...
----------------------------------------
🎉 KẾT QUẢ CUỐI CÙNG (DATASET KHỔNG LỒ):
label
0    15907
1     6168
Name: count, dtype: int64
-> Tỉ lệ: 15907 Tin Thật vs 6168 Tin Giả
----------------------------------------
Tổng cộng: 22075 dòng dữ liệu.


In [3]:
import torch
from torch.utils.data import Dataset, DataLoader
# SỬA LỖI Ở DÒNG NÀY: Bỏ AdamW khỏi transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification
# VÀ IMPORT ADAMW TỪ TORCH
from torch.optim import AdamW

from sklearn.model_selection import train_test_split
from pyvi import ViTokenizer
import re
import string
import numpy as np

# 1. CẤU HÌNH
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Đang sử dụng thiết bị: {device}")
BATCH_SIZE = 16
EPOCHS = 3

# 2. TIỀN XỬ LÝ DỮ LIỆU (Chạy trên biến df vừa gộp)
print("Đang tiền xử lý và tách từ (việc này mất khoảng 1-2 phút)...")

def clean_and_segment(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    return ViTokenizer.tokenize(text)

# Áp dụng cho toàn bộ dữ liệu
# Lưu ý: Nếu df của bạn lớn, bước này sẽ mất chút thời gian
df['text_processed'] = df['text'].apply(clean_and_segment)

# Chia tập Train (80%) và Validation (20%)
X_train, X_val, y_train, y_val = train_test_split(df['text_processed'], df['label'], test_size=0.2, random_state=42)

# 3. CHUẨN BỊ DATASET CHO PYTORCH
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")

class FakeNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]

        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            return_attention_mask=True,
            return_tensors='pt',
            truncation=True
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Tạo DataLoader
train_dataset = FakeNewsDataset(X_train.to_numpy(), y_train.to_numpy(), tokenizer)
val_dataset = FakeNewsDataset(X_val.to_numpy(), y_val.to_numpy(), tokenizer)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

# 4. KHỞI TẠO MODEL
print("Đang tải model PhoBERT...")
model = AutoModelForSequenceClassification.from_pretrained("vinai/phobert-base", num_labels=2)
model = model.to(device)
optimizer = AdamW(model.parameters(), lr=2e-5)

# 5. VÒNG LẶP TRAINING
def train_epoch(model, data_loader, optimizer, device):
    model.train()
    losses = []
    correct_predictions = 0

    for d in data_loader:
        input_ids = d["input_ids"].to(device)
        attention_mask = d["attention_mask"].to(device)
        labels = d["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        logits = outputs.logits

        _, preds = torch.max(logits, dim=1)
        correct_predictions += torch.sum(preds == labels)
        losses.append(loss.item())

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    return correct_predictions.double() / len(data_loader.dataset), sum(losses) / len(data_loader)

def eval_model(model, data_loader, device):
    model.eval()
    correct_predictions = 0

    with torch.no_grad():
        for d in data_loader:
            input_ids = d["input_ids"].to(device)
            attention_mask = d["attention_mask"].to(device)
            labels = d["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            _, preds = torch.max(outputs.logits, dim=1)
            correct_predictions += torch.sum(preds == labels)

    return correct_predictions.double() / len(data_loader.dataset)

print("\n=== BẮT ĐẦU TRAINING ===")
for epoch in range(EPOCHS):
    print(f'Epoch {epoch + 1}/{EPOCHS}')
    train_acc, train_loss = train_epoch(model, train_loader, optimizer, device)
    print(f'Train loss: {train_loss:.4f}, Accuracy: {train_acc:.4f}')

    val_acc = eval_model(model, val_loader, device)
    print(f'Validation Accuracy: {val_acc:.4f}')
    print('-' * 20)

Đang sử dụng thiết bị: cuda
Đang tiền xử lý và tách từ (việc này mất khoảng 1-2 phút)...


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

2026-01-03 15:35:37.984103: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767454538.177283      25 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767454538.230599      25 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767454538.691885      25 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767454538.691926      25 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767454538.691929      25 computation_placer.cc:177] computation placer alr

pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/phobert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]


=== BẮT ĐẦU TRAINING ===
Epoch 1/3
Train loss: 0.3181, Accuracy: 0.8528
Validation Accuracy: 0.9051
--------------------
Epoch 2/3
Train loss: 0.1926, Accuracy: 0.9223
Validation Accuracy: 0.9219
--------------------
Epoch 3/3
Train loss: 0.1215, Accuracy: 0.9545
Validation Accuracy: 0.9257
--------------------


In [4]:
import os

# TRÊN KAGGLE, ĐƯỜNG DẪN BẮT BUỘC PHẢI LÀ /kaggle/working/
save_path = '/kaggle/working/PhoBERT_FakeNews_Final'

if not os.path.exists(save_path):
    os.makedirs(save_path)

print(f"Đang lưu model vào {save_path}...")
# Lưu Model và Tokenizer
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("✅ Đã lưu xong! Hãy kiểm tra phần Output.")

Đang lưu model vào /kaggle/working/PhoBERT_FakeNews_Final...
✅ Đã lưu xong! Hãy kiểm tra phần Output.
